# EasyRAG Observability And Fallbacks

## What you will do

- inspect retrieval and generation metadata as observability signals
- trigger fallback behavior on purpose
- compare healthy and degraded runs using the same output schema

## Why this matters

Production behavior is easier to trust when weak paths are visible instead of silent. EasyRAG already exposes several useful runtime signals for that job.


## Source code anchors

- `easyrag.rag.retrieval.pipeline.execute_query`
- `easyrag.rag.generation.pipeline.generate_answer`
- `easyrag.rag.retrieval.hydration.detect_vector_backend`
- `easyrag.rag.generation.synthesis.synthesize_answer`


In [ ]:
# ruff: noqa: E402,F401,F811
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "easyrag").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

import json
import os
import tempfile
import time
from pathlib import Path
from pprint import pprint
from unittest import mock

from easyrag.rag import AnswerParam, EasyRAG, EvalCase, QueryParam
from easyrag.support.async_utils import run_sync
from notebooks._utils import (
    managed_demo_rag,
    print_json as _print_json,
    production_backends_ready,
    provider_ready as can_use_openai_compatible_models,
    skip_message,
    stub_embedding as _stub_embedding,
    stub_query_model as _stub_query_model,
    stub_reranker as _stub_reranker,
)


## Deterministic path

We first run a healthy retrieval-and-answer flow so the normal metadata baseline is explicit.


In [ ]:
obs_tmp = tempfile.TemporaryDirectory()
obs_rag = EasyRAG(
    working_dir=obs_tmp.name,
    workspace="observability-demo",
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
)
run_sync(obs_rag.initialize_storages())
run_sync(obs_rag.ainsert("# Retrieval\nEasyRAG uses grounded retrieval and query rewrite.\n", ids=["doc::retrieval"], file_paths=["docs/retrieval.md"]))
healthy_query = run_sync(obs_rag.aquery("How does EasyRAG use query rewrite?", QueryParam(mode="hybrid", enable_rerank=False, chunk_top_k=3)))
healthy_answer = run_sync(obs_rag.aanswer("How does EasyRAG use query rewrite?", QueryParam(mode="hybrid", enable_rerank=False, chunk_top_k=3), AnswerParam()))
_print_json({
    "healthy_query_metadata": healthy_query.metadata,
    "healthy_answer_metadata": healthy_answer.metadata,
})


## Output inspection

The next cell forces fallback behavior in both retrieval and generation. Retrieval falls back to token overlap when embeddings fail. Generation falls back to the built-in grounded answer when the answer model raises an error.


In [ ]:
def failing_embedding(texts):
    raise RuntimeError("embedding backend unavailable")


def failing_answer_model(prompt: str, **kwargs):
    raise RuntimeError("answer model unavailable")

fallback_tmp = tempfile.TemporaryDirectory()
fallback_rag = EasyRAG(
    working_dir=fallback_tmp.name,
    workspace="observability-fallback-demo",
    embedding_func=failing_embedding,
    query_model_func=_stub_query_model,
    answer_model_func=failing_answer_model,
)
run_sync(fallback_rag.initialize_storages())
run_sync(fallback_rag.ainsert("# Retrieval\nretrieval retrieval keeps the fallback demo visible.\n", ids=["doc::retrieval"], file_paths=["docs/retrieval.md"]))
fallback_query = run_sync(fallback_rag.aquery("retrieval", QueryParam(mode="naive", rewrite_enabled=False, mqe_enabled=False)))
fallback_answer = run_sync(fallback_rag.aanswer("How does EasyRAG use retrieval?", QueryParam(mode="naive", rewrite_enabled=False, mqe_enabled=False), AnswerParam()))
_print_json({
    "fallback_query_metadata": fallback_query.metadata,
    "fallback_answer_metadata": fallback_answer.metadata,
    "fallback_answer": fallback_answer.answer,
})


## Provider-backed path

When providers are configured, the same metadata fields still act as the observability surface. The optional path prints those fields directly.


In [ ]:
if not can_use_openai_compatible_models():
    print("Skipping provider-backed path because OPENAI-compatible config is not set.")
else:
    provider_tmp = tempfile.TemporaryDirectory()
    provider_rag = EasyRAG(working_dir=provider_tmp.name, workspace="provider-observability-demo")
    try:
        run_sync(provider_rag.initialize_storages())
        run_sync(provider_rag.ainsert("# Retrieval\nGrounded retrieval stays inspectable.\n", ids=["doc::retrieval"], file_paths=["docs/retrieval.md"]))
        provider_result = run_sync(provider_rag.aquery("grounded retrieval", QueryParam(mode="hybrid", chunk_top_k=2)))
        _print_json(provider_result.metadata)
    except Exception as exc:
        print(f"Provider-backed path was skipped due to runtime error: {exc}")
    finally:
        try:
            run_sync(provider_rag.finalize_storages())
        finally:
            provider_tmp.cleanup()


## What changed and why

Observability becomes useful when the same metadata surface covers both healthy and degraded runs. `stage_timings_ms` shows where time was spent. `candidate_counts` shows where evidence disappeared. `vector_backend` and `fallback_used` show whether the retrieval path degraded. Answer metadata shows whether synthesis abstained or fell back gracefully.


In [ ]:
run_sync(obs_rag.finalize_storages())
obs_tmp.cleanup()
run_sync(fallback_rag.finalize_storages())
fallback_tmp.cleanup()
print("Cleaned up the observability workspaces.")


## Next steps

- Read [08-system-architecture-overview.md](../../docs/08-system-architecture-overview.md) for the chapter-level architecture map.
- Revisit [04_07_retrieval_failure_cases_and_debugging.ipynb](../04_retrieval/04_07_retrieval_failure_cases_and_debugging.ipynb) and [05_05_generation_failures_and_guardrails.ipynb](../05_generation/05_05_generation_failures_and_guardrails.ipynb) for narrower stage-specific debugging views.
- Continue with [07-optimization-overview.md](../../docs/07-optimization-overview.md) once you want to turn these signals into concrete tuning priorities.
